In [1]:
import pandas as pd
import numpy as np
# from scipy.signal import argrelextrema
import plotly.graph_objects as go
from plotly.subplots import make_subplots
# from sklearn.linear_model import LinearRegression
# from datetime import datetime, date, timedelta, timezone, time
import talib.abstract as ta
# import seaborn as sns
# import matplotlib.pyplot as plt
import freqtrade.vendor.qtpylib.indicators as qtpylib
from freqtrade.strategy import merge_informative_pair
import os
from pathlib import Path
from freqtrade.configuration import Configuration
from freqtrade.data.btanalysis import load_backtest_data

In [2]:
project_root = "."
i=0
try:
    os.chdirdir(project_root)
    assert Path('.gitignore').is_file()
except:
    while i<4 and (not Path('.gitignore').is_file()):
        os.chdir(Path(Path.cwd(), '../'))
        i+=1
    project_root = Path.cwd()
print(Path.cwd())

/home/mo/Repositories/trade


In [ ]:
ticker = 'BTC'

config = Configuration.from_files(["user_data/atlas_engine_test.json"])
config["timeframe"] = "1h"
config["strategy"] = "AtlasEngine"
data_location = config["datadir"]
pair = f"{ticker}/USDT:USDT"

backtest_dir = config["user_data_dir"] / "backtest_results"
trades = load_backtest_data(backtest_dir)
trades['color'] = np.where(trades.profit_abs >= 0, 'green', 'red')

base_url = "/home/mo/Repositories/trade/user_data/data/bybit/futures/"
dataframe_15m = pd.read_feather(f"{base_url}{ticker}_USDT_USDT-15m-futures.feather")
dataframe_1h = pd.read_feather(f"{base_url}{ticker}_USDT_USDT-1h-futures.feather")
dataframe_4h = pd.read_feather(f"{base_url}{ticker}_USDT_USDT-4h-futures.feather")
dataframe_1d = pd.read_feather(f"{base_url}{ticker}_USDT_USDT-1d-futures.feather")

In [52]:
def extract_features(dataframe, c1, c2, col, name, tt, pt, d="forward"):
    
    df = dataframe.copy()

    starts = df.loc[c1].reset_index().rename(columns={"index": "start"})
    ends   = df.loc[c2].reset_index().rename(columns={"index": "end"})

    if starts.empty or ends.empty:
        dataframe[name] = np.nan
        dataframe[f"{name}_index_dist"] = np.nan
        dataframe[f"{name}_price_dist"] = np.nan
        return dataframe

    pairs = pd.merge_asof(
        starts[['start']].sort_values("start"),
        ends[['end']].sort_values("end"),
        left_on="start",
        right_on="end",
        direction=d,
    ).dropna()[["start", "end"]]

    if pairs.empty:
        dataframe[name] = np.nan
        dataframe[f"{name}_index_dist"] = np.nan
        dataframe[f"{name}_price_dist"] = np.nan
        return dataframe

    intervals = pd.IntervalIndex.from_arrays(pairs["start"], pairs["end"], closed="both")
    df["range_id"] = pd.cut(df.index, intervals)

    e = 'max' if col == 'high' else 'min'
    group = df.groupby("range_id", observed=True)
    df[name] = group[col].transform(e)

    df[f"{name}_index_dist"] = group.cumcount() + 1
    df[f"{name}_index_dist"] = group[f"{name}_index_dist"].transform('max')

    hi = group['high'].transform('max')
    lo = group['low'].transform('min')
    df[f"{name}_price_dist"] = np.abs(hi - lo)

    df.loc[
        (df[col] != df[name]),
        [name, f"{name}_index_dist", f"{name}_price_dist"]
    ] = np.nan

    dataframe = dataframe.merge(df[[name, f"{name}_index_dist", f"{name}_price_dist"]],
                        left_index=True, right_index=True, how="left")

    c1 = (dataframe[f'{name}_index_dist'] >= tt)
    price_threshold = dataframe[f'{name}_price_dist'].quantile(pt)
    c2 = (dataframe[f'{name}_price_dist'] >= price_threshold)
    dataframe[name] = np.where((c1 & c2), dataframe[name], np.nan)

    # dataframe[name] = dataframe[name].ffill()
    
    return dataframe

In [53]:
def populate_features(dataframe, rsi_high=70, rsi_low=30, tt=0, pt=0):

    dataframe["rsi"] = ta.RSI(dataframe["close"], timeperiod=14)

    c1 = qtpylib.crossed_above(dataframe["rsi"], rsi_high)
    c2 = qtpylib.crossed_below(dataframe["rsi"], rsi_high)
    dataframe = extract_features(dataframe, c1, c2, "high", "max_high", tt=tt, pt=pt)
    dataframe = extract_features(dataframe, c2, c1, "low", "min_high", tt=tt, pt=pt)

    c1 = qtpylib.crossed_below(dataframe["rsi"], rsi_low)
    c2 = qtpylib.crossed_above(dataframe["rsi"], rsi_low)
    dataframe = extract_features(dataframe, c1, c2, "low", "min_low", tt=tt, pt=pt)
    dataframe = extract_features(dataframe, c2, c1, "high", "max_low", tt=tt, pt=pt)

    dataframe.loc[dataframe['max_high']==dataframe['high'],"cat"] = 'H'
    dataframe.loc[dataframe['min_low']==dataframe['low'],"cat"] = 'L'
    dataframe['cat'] = dataframe['cat'].ffill()

    return dataframe

In [6]:
def plot(dataframe, trades, candle=False, row_heights=[0.55, 0.15, 0.15, 0.15], p1=[], p2=[], p3=[], p4=[]):
    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=True,
        row_heights=row_heights,
        vertical_spacing=0.05
    )

    if candle:
        fig.add_trace(go.Candlestick(
            x=dataframe['date'],
            open=dataframe['open'],
            high=dataframe['high'],
            low=dataframe['low'],
            close=dataframe['close'],
            name='Price'
        ))
    
    for col, mode, color in p1:
        fig.add_trace(
            go.Scatter(
                x=dataframe.date.values,
                y=dataframe[col].values,
                mode=mode,
                line=dict(color=color),
                name=col
            ),
            row=1, col=1
        )

    if not trades.empty:
        fig.add_trace(
            go.Scatter(
                x=trades.open_date,
                y=trades.open_rate,
                mode='markers',
                name="open date",
                marker=dict(
                    color='orange',
                    symbol="square-open",
                    size=10,
                    line=dict(width=3),
                ),
            ),
            row=1, col=1
        )
        
        fig.add_trace(
            go.Scatter(
                x=trades.close_date,
                y=trades.close_rate,
                mode='markers',
                name="trades",
                marker=dict(
                    color=trades.color,
                    symbol="square-open",
                    size=10,
                    line=dict(width=3),
                )
            ),
            row=1, col=1
        )

    for col, mode, color in p2:
        fig.add_trace(
            go.Scatter(
                x=dataframe.date.values,
                y=dataframe[col].values,
                mode=mode,
                line=dict(color=color),
                name=col
            ),
            row=2, col=1
        )

    fig.add_hline(70, row=2, col=1)
    fig.add_hline(30, row=2, col=1)

    for col, mode, color in p3:
        fig.add_trace(
            go.Scatter(
                x=dataframe.date.values,
                y=dataframe[col].values,
                mode=mode,
                line=dict(color=color),
                name=col
            ),
            row=3, col=1
        )

    fig.add_hline(70, row=3, col=1)
    fig.add_hline(30, row=3, col=1)

    for col, mode, color in p4:
        fig.add_trace(
            go.Scatter(
                x=dataframe.date.values,
                y=dataframe[col].values,
                mode=mode,
                line=dict(color=color),
                name=col
            ),
            row=4, col=1
        )

    fig.add_hline(70, row=4, col=1)
    fig.add_hline(30, row=4, col=1)

    fig.update_layout(
        height=800, 
        showlegend=True,
        xaxis_rangeslider_visible=False
    )
    
    fig.show()

In [13]:
def populate_entry_trend(dataframe):

    dataframe.loc[
        (   (dataframe['min_low'] > dataframe['min_low_1d']) & # Guard
            (dataframe['cat_1d'] == "L") & # Guard
            (dataframe['cat'] == "L") & # Guard
            (dataframe['rsi'] > 30) & # Guard
            (qtpylib.crossed_above(dataframe["close"], dataframe['max_high'])) # Trigger
        ),
        "enter_long"
    ] = 1

    dataframe.loc[
        (
            (dataframe['max_high'] < dataframe['max_high_1d']) & # Guard
            (dataframe['cat_1d'] == "H") & # Guard
            (dataframe['cat'] == "H") & # Guard
            (dataframe['rsi'] < 70) & # Guard
            (qtpylib.crossed_below(dataframe["close"], dataframe['max_high'])) # Trigger
        ),
        "enter_short"
    ] = 1

    return dataframe

In [ ]:
dataframe = dataframe_15m.copy()
dataframe = populate_features(dataframe, rsi_high=70, rsi_low=30, tt=3, pt=0.8)

informative = dataframe_1h.copy()
informative = populate_features(informative, rsi_high=70, rsi_low=30, tt=3, pt=0.8)
dataframe = merge_informative_pair(dataframe, informative, config["timeframe"], '1h', ffill=False)

informative = dataframe_4h.copy()
informative = populate_features(informative, rsi_high=70, rsi_low=30, tt=3, pt=0.8)
dataframe = merge_informative_pair(dataframe, informative, config["timeframe"], '4h', ffill=False)

informative = dataframe_1d.copy()
informative = populate_features(informative, rsi_high=70, rsi_low=30, tt=2, pt=0.8)
dataframe = merge_informative_pair(dataframe, informative, config["timeframe"], '1d', ffill=False)


In [55]:
dataframe = populate_entry_trend(dataframe)

In [56]:
start = '2023-12-01'
end = '2024-03-30'
trades_red = trades.loc[
    (trades['pair'] == pair) & 
    (trades.open_date > start) & 
    (trades.open_date < end) &
    (trades.is_short == False)
]
data_red = dataframe.loc[(dataframe.date > start) & (dataframe.date < end)]

In [57]:
plot(
    data_red,
    trades_red,
    p1=[
        ('close','lines','blue'),
        ('max_high','markers','orange'),
        # ('max_high_1h','markers','yellow'),
        # ('max_high_4h','markers','red'),
        # ('max_high_1d','markers','purple')
    ], 
    p2=[
        (f'rsi','lines','blue'),
    ],
    p3=[
        (f'rsi_4h','lines','blue'),
    ],
    p4=[
        (f'rsi_1d','lines','blue'),
    ]
)

In [ ]:
# docker-compose run --rm atlas_engine_test backtesting --strategy AtlasEnginePlus --config user_data/atlas_engine_test.json --timerange 20231218- --export trades